In [ ]:
try:
    get_ipython().run_line_magic('load_ext' , 'autoreload')
    get_ipython().run_line_magic('autoreload', 2)
except:
    pass

import scipy.io as sio
import datetime
import h5py
import glob
import fnmatch
import os

import logging
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.ndimage import gaussian_filter
from tifffile.tifffile import imwrite

from extract_models import extract_2d
import caiman as cm
from caiman.utils.visualization import plot_contours, nb_plot_contour, nb_view_patches3d
import caiman.source_extraction.cnmf as cnmf
from caiman.paths import caiman_datadir

import bokeh.plotting as bpl
bpl.output_notebook()

logging.basicConfig(format=
                          "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                    # filename="/tmp/caiman.log",
                    #level=logging.INFO,
                    )



In [ ]:
type(int(np.round(0.9)))

In [ ]:
only_init = False

sly = slice(0, 51, 1)
slx = slice(40, 181, 1) 
slz = slice(0, 70, 1)
indices_ex = [slx, sly, slz]
indices_ex = [slice(None), slice(None), slice(None)]

p = 0                   # order of the autoregressive system - 0 from carl's code
merge_thresh = 0.9
gSig = [4, 4, 4]  # gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
nb = 2                  # temporal global background components - TUNE

do_patches = False      # flag for processing in patches or not - turn on or off - Not used in Matlab
if do_patches:          # PROCESS IN PATCHES AND THEN COMBINE
    rf = 60             # half size of each patch
    stride_cnmf = 40          # overlap between patches
    p_patch = p
    nb_patch = nb
    k = 3              # number of components in each patch
    indices_ex = [slice(None), slice(None), slice(None)]
else:                   # PROCESS THE WHOLE FOV AT ONCE
    rf = None           # setting these parameters to None
    stride_cnmf = None       # will run CNMF on the whole FOV
    p_patch = p
    nb_patch = nb
    k = 2              # number of neurons expected (in the whole FOV) - 40 from Carl's Code, seems to be too many

###
fr = 5.08 #0.6193  #9.8465 frame period so 1000 / (9.8465 *(113+51)) # approximate frame rate of data - CONFIRMED FPS
decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
dxy = [1.33155792277, 1.33155792277, 0.16666666666666666] #for .751 um pixels # pixels per micron 

tsub = 1                # temporal downsampling
ssub = 1               # spatial downsampling

method_init = 'greedy_roi' #'graph_nmf' #'greedy_roi' #python Caiman defaults to greedy_roi, carl's code uses sparse_nmf, but sparse_nmf runs MUCH slower
sigma_smooth_snmf = (0.5, 0.5, 0.5)
perc_baseline_snmf = 50

se = np.ones((3,)*len(gSig), dtype=np.uint8)  #se = np.ones((3,3,1), dtype=np.uint8)
update_background_components = True   #use this??

fudge_factor = 0.96        # (default is 0.96; Carl's value = 1) -- bias correction factor for discrete time constants
ITER = 5                # (default is 2; Carl's value=5) -- block coordinate descent iterations
bas_nonneg = False #True

rolling_sum = True

min_SNR = 0      # accept components with that peak-SNR or higher (if above this, acept)
SNR_lowest = 0         # minimum SNR for accepted components (if below this, reject)
rval_lowest = 0  # 0.6  # space correlation threshold (if above this, accept)
rval_thr = 0  # 0.6  # space correlation threshold (if above this, accept)
use_cnn = False      # use the CNN classifier affects if 2 below params are used
min_cnn_thr = 0  # if cnn classifier predicts below this value, reject
cnn_lowest = 0.1   # neurons with cnn probability lower than this value are rejected

strides_mc = (24, 24, 6)
overlaps_mc = (12, 12, 2)
max_shifts_mc = (4, 4, 2)
max_deviation_rigid = 5
pw_rigid = False
is3D_mc = True
indices_mc = (slice(None), slice(None), slice(None)) #if is3d is true for motion correction, will overwrite with nones and will lose indices_ex


opts_dict = {'strides': strides_mc,    # start a new patch for pw-rigid motion correction every x pixels
            'overlaps': overlaps_mc,   # overlap between pathes (size of patch strides+overlaps)
            'max_shifts': max_shifts_mc,   # maximum allowed rigid shifts (in pixels)
            'max_deviation_rigid': max_deviation_rigid,  # maximum shifts deviation allowed for patch with respect to rigid shifts
            'pw_rigid': pw_rigid,         # flag for performing non-rigid motion correction
            'is3D': is3D_mc,
            'fr': fr,
            'p': p,
            'nb': nb,
            'merge_thr': merge_thresh,
            'rf': rf,
            'indices': indices_mc,  #for some reason indices_mc is causing error, maybe needs list for mc and tuple for extraction?
            'K': k, 
            'gSig': gSig,
            'stride': stride_cnmf,
            'method_init': method_init,
            'sigma_smooth_snmf': sigma_smooth_snmf,
            'perc_baseline_snmf': perc_baseline_snmf,
            #'dims': dims,
            'dxy': dxy,
            'decay_time': decay_time,
            'bas_nonneg': bas_nonneg,
            'p_patch': p_patch,
            'nb_patch': nb_patch,
            'se': se, 
            'rolling_sum': rolling_sum,
            'only_init': only_init,
            'ssub': ssub,
            'tsub': tsub,
            'SNR_lowest': SNR_lowest, 
            'min_SNR': min_SNR,
            'rval_thr': rval_thr,
            'use_cnn': use_cnn,
            'min_cnn_thr': min_cnn_thr,
            'cnn_lowest': cnn_lowest, 
            'ITER': ITER, 
            'fudge_factor': fudge_factor, 
            'cnn_lowest': cnn_lowest, 
            'update_background_components': update_background_components}


In [ ]:


anatomical_stack = True
planar_extraction = True 
do_motion_correction = True
do_refit = True
dimin = [80, 164, 140, 256] 
flyback = 51

recordingIDs = ['20230627-2']
data_path_prefix = '/Users/wienecke/Documents/ambrose/leprechaunMat/'


for recordingID in recordingIDs:
  
  tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #create extraction ID, one for each recordingID

  path_date_stack = data_path_prefix + recordingID + '_D05_syt7f_018_syt7f/'
  path_to_stacks = []

  if anatomical_stack:
    stack_filename_template = recordingID.split('-')[0] + '_' + recordingID.split('-')[1] + '_hires_.tif'
  else:
    stack_filename_template = recordingID + '*_trial_*.tif'

  for f in os.listdir(path_date_stack):
    if fnmatch.fnmatch(f,stack_filename_template):
      tmp = path_date_stack + f
      path_to_stacks.append(tmp)

  for f in path_to_stacks:

    print(f)
    Y = cm.load(f) #scan image tif files are loaded (and presumably saved) as t z y x
    Y = Y.reshape(dimin[0], dimin[1], dimin[2], dimin[3])
    Y = Y[:,:-flyback,:,:] #crop flyback frames

    Y[20,:,:,:].play(magnification=2) 

    cc, dview, n_processes = cm.cluster.setup_cluster(backend='local', n_processes=None, single_thread=False)


In [ ]:
    cc, dview, n_processes = cm.cluster.setup_cluster(backend='local', n_processes=None, single_thread=False)


In [ ]:
anatomical_stack = True
planar_extraction = True 
do_motion_correction = True
do_refit = True
dimin = [80, 164, 140, 256] 
flyback = 51

recordingIDs = ['20230627-2']
data_path_prefix = '/Users/wienecke/Documents/ambrose/leprechaunMat/'

for recordingID in recordingIDs:
  
  tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #create extraction ID, one for each recordingID

  path_date_stack = data_path_prefix + recordingID + '_D05_syt7f_018_syt7f/'
  path_to_stacks = []

  if anatomical_stack:
    stack_filename_template = recordingID.split('-')[0] + '_' + recordingID.split('-')[1] + '_hires_.tif'
  else:
    stack_filename_template = recordingID + '*_trial_*.tif'

  for f in os.listdir(path_date_stack):
    if fnmatch.fnmatch(f,stack_filename_template):
      tmp = path_date_stack + f
      path_to_stacks.append(tmp)

  for f in path_to_stacks:

    print(f)
    Y = cm.load(f) #scan image tif files are loaded (and presumably saved) as t z y x
    Y = Y.reshape(dimin[0], dimin[1], dimin[2], dimin[3])
    Y = Y[:,:-flyback,:,:] #crop flyback frames

    #Y[20,:,:,:].play(magnification=2) 

    Y = np.transpose(Y, (0, 3, 2, 1)) #put in order t x y z 

    Y = Y - np.min(Y) #make minimum zero (not certain this is the place for this, or whether it should occur at all)

    fname = [f[:(len(f)-4)] + 'caimanreg_.tif']
    fname_save = fname[0][:(len(f)-4)] + tmpdate + '_caimanrois_.mat'

    imwrite(fname[0], Y) #write as t x y z
    dims = Y.shape[1:]
    
    fug = extract_2d(opts_dict, fname, do_motion_correction, planar_extraction, do_refit, indices_ex, fname_save)



